# Práctica: Índice Inverso, Lematización y Bolsas de Palabras
**Datos Masivos I**

## 1. Importaciones y Configuración

In [ ]:
import math
import time
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from sklearn.datasets import fetch_20newsgroups
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

nltk.download('punkt',                      quiet=True)
nltk.download('punkt_tab',                  quiet=True)
nltk.download('wordnet',                    quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('omw-1.4',                    quiet=True)

lemmatizer = WordNetLemmatizer()
print('Librerías cargadas correctamente.')

## 2. Carga de Datos – 20 Newsgroups

In [ ]:
newsgroups_raw   = fetch_20newsgroups(subset='train')
newsgroups_clean = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

orig       = newsgroups_clean.data
etiquetas  = newsgroups_raw.target       # índice de categoría por documento
categorias = newsgroups_raw.target_names

print(f'Documentos cargados : {len(orig)}')
print(f'Categorías          : {len(categorias)}')
for cat in categorias:
    print(f'  - {cat}')

In [ ]:
# Vista previa de un documento sin procesar
idx = 0
print(f'Categoría : {categorias[etiquetas[idx]]}')
print(f'Texto     :\n{orig[idx][:500]}...')

## 3. Stopwords

In [ ]:
with open('datitos.txt', 'r', encoding='utf-8') as f:
    datitos = {line.strip() for line in f if line.strip()}

print(f'Stopwords cargadas: {len(datitos)}')
print(f'Ejemplo            : {list(datitos)[:10]}')

## 4. Preprocesamiento de Texto

El pipeline de limpieza aplica los siguientes pasos a cada documento:
1. Tokenización y conversión a minúsculas
2. Filtrado de tokens no alfabéticos
3. **POS tagging en batch** (una sola llamada por documento)
4. Lematización guiada por la etiqueta gramatical
5. Eliminación de stopwords

In [ ]:
def get_wordnet_pos(tag):
    """Convierte etiqueta Penn Treebank al formato de WordNet."""
    tag_dict = {'J': wordnet.ADJ, 'N': wordnet.NOUN, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_dict.get(tag[0].upper(), wordnet.NOUN)


def lematizar(texto):
    if not texto:
        return ''
    tokens = [w for w in word_tokenize(texto.lower()) if w.isalpha()]
    # pos_tag recibe todos los tokens de golpe (mucho más rápido que uno por uno)
    lemas = [lemmatizer.lemmatize(w, get_wordnet_pos(pos)) for w, pos in nltk.pos_tag(tokens)]
    return ' '.join(lemas)


def mochar(texto, stopwords):
    return [w for w in texto.split() if w not in stopwords]

In [ ]:
texto_ejemplo = 'The dogs were running quickly in different NASA missions. Studies were changing rapidly!'
lematizado    = lematizar(texto_ejemplo)
filtrado      = mochar(lematizado, datitos)

print(f'Original  : {texto_ejemplo}')
print(f'Lematizado: {lematizado}')
print(f'Filtrado  : {filtrado}')

## 5. Construcción del Corpus Limpio

In [ ]:
procesados = []
vacios     = 0

for texto in orig:
    tokens = mochar(lematizar(texto), datitos)
    if tokens:
        procesados.append(' '.join(tokens))
    else:
        vacios += 1

print(f'Documentos procesados         : {len(procesados)}')
print(f'Documentos vacíos descartados : {vacios}')

## 6. Frecuencias del Corpus

Se calculan dos métricas por término:
- **freq_total**: cuántas veces aparece en todo el corpus
- **freq_documental**: en cuántos documentos distintos aparece

In [ ]:
def calcular_frecuencias(documentos):
    freq_total      = defaultdict(int)
    freq_documental = defaultdict(int)
    for doc in documentos:
        palabras = doc.split()
        for p in palabras:
            freq_total[p] += 1
        for p in set(palabras):
            freq_documental[p] += 1
    return dict(freq_total), dict(freq_documental)


freq_total, freq_documental = calcular_frecuencias(procesados)

TOP_N        = 5000
palabras_max = sorted(freq_total, key=freq_total.get, reverse=True)[:TOP_N]

print(f'Vocabulario total   : {len(freq_total):,} palabras únicas')
print(f'Vocabulario reducido: {TOP_N:,} palabras (top por frecuencia)')
print()
print(f'{"Palabra":<20} {"Freq. total":>12} {"Freq. doc.":>12}')
print('-' * 46)
for p in palabras_max[:10]:
    print(f'{p:<20} {freq_total[p]:>12,} {freq_documental[p]:>12,}')

## 7. Bolsas de Palabras

Cada documento se representa como una lista dispersa de pares `(palabra, frecuencia)`,
incluyendo únicamente los términos que pertenecen al vocabulario reducido y que aparecen en el documento.

In [ ]:
def calcular_bolsas(documentos, freq_total, n=5000):
    vocab = set(sorted(freq_total, key=freq_total.get, reverse=True)[:n])
    resultado = []
    for doc in documentos:
        conteo = Counter(doc.split())
        bolsa  = [(p, c) for p, c in conteo.items() if p in vocab]
        resultado.append(bolsa)
    return resultado


conteodocs = calcular_bolsas(procesados, freq_total)

print(f'Bolsas generadas: {len(conteodocs)}')
print(f'Ejemplo – doc 0 (top 8 términos por frecuencia):')
print(sorted(conteodocs[0], key=lambda x: x[1], reverse=True)[:8])

## 8. Vocabulario

Se construye el vocabulario formal asignando a cada palabra un **ID numérico** y su valor **IDF**:

$$idf(t) = \log\!\left(\frac{N}{df(t)}\right)$$

donde $N$ es el total de documentos y $df(t)$ es cuántos documentos contienen el término $t$.

In [ ]:
N = len(procesados)

vocab_idx   = {p: i for i, p in enumerate(palabras_max)}   # palabra → posición en el vector
vocabulario = {
    p: {
        'id':         i,
        'freq_total': freq_total[p],
        'freq_doc':   freq_documental[p],
        'idf':        math.log(N / freq_documental[p])
    }
    for i, p in enumerate(palabras_max)
}

print(f'Tamaño del vocabulario: {len(vocabulario):,} términos')
print()
print(f'{"Palabra":<20} {"ID":>6} {"Freq.Total":>11} {"Freq.Doc":>10} {"IDF":>8}')
print('-' * 60)
for p in palabras_max[:10]:
    v = vocabulario[p]
    print(f'{p:<20} {v["id"]:>6} {v["freq_total"]:>11,} {v["freq_doc"]:>10,} {v["idf"]:>8.4f}')

## 9. Representación Vectorial

Cada bolsa de palabras se convierte en un vector de dimensión fija (= tamaño del vocabulario).

- **Vector de conteos**: la posición $i$ contiene $tf(t_i, d)$
- **Vector TF-IDF**: la posición $i$ contiene $tf(t_i, d) \times idf(t_i)$

Se pre-normaliza la matriz TF-IDF (norma $L_2 = 1$) para que la similitud coseno sea un simple producto punto.

In [ ]:
def bolsa_a_vector_conteo(bolsa, vocab_idx):
    v = np.zeros(len(vocab_idx), dtype=np.float32)
    for p, c in bolsa:
        if p in vocab_idx:
            v[vocab_idx[p]] = c
    return v


def bolsa_a_vector_tfidf(bolsa, vocab_idx, vocabulario):
    v = np.zeros(len(vocab_idx), dtype=np.float32)
    for p, c in bolsa:
        if p in vocab_idx:
            v[vocab_idx[p]] = c * vocabulario[p]['idf']
    return v

In [ ]:
print('Construyendo matriz TF-IDF...')
t0 = time.time()

tfidf_matrix = np.zeros((len(conteodocs), len(vocab_idx)), dtype=np.float32)
for i, bolsa in enumerate(conteodocs):
    for p, c in bolsa:
        if p in vocab_idx:
            tfidf_matrix[i, vocab_idx[p]] = c * vocabulario[p]['idf']

# Pre-normalizar: coseno = producto punto entre vectores ya normalizados
norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1
tfidf_norm = (tfidf_matrix / norms).astype(np.float32)

print(f'Listo en {time.time()-t0:.1f} s')
print(f'Dimensiones : {tfidf_matrix.shape[0]:,} docs × {tfidf_matrix.shape[1]:,} dims')
print(f'Memoria     : {tfidf_matrix.nbytes / 1e6:.0f} MB')

In [ ]:
# Demostración con dos documentos
i, j = 0, 1
v0_tfidf = bolsa_a_vector_tfidf(conteodocs[i], vocab_idx, vocabulario)
v0_cont  = bolsa_a_vector_conteo(conteodocs[i], vocab_idx)

print(f'Doc {i} – categoría: {categorias[etiquetas[i]]}')
print(f'  Entradas no-cero en vector TF-IDF : {np.count_nonzero(v0_tfidf)}')
print(f'  Entradas no-cero en vector conteo : {np.count_nonzero(v0_cont)}')
print(f'  Norma TF-IDF                      : {np.linalg.norm(v0_tfidf):.4f}')

## 10. Medidas de Similitud

Se implementan tres medidas para comparar documentos:

| Medida | Fórmula | Opera sobre |
|--------|---------|-------------|
| **Coseno** | $\cos(x,y)=\frac{x\cdot y}{\|x\|\|y\|}$ | Vectores TF-IDF |
| **Jaccard** | $J(A,B)=\frac{|A\cap B|}{|A\cup B|}$ | Conjuntos de palabras |
| **MinMax** | $S(x,y)=\frac{\sum_i\min(x_i,y_i)}{\sum_i\max(x_i,y_i)}$ | Vectores de conteos |

In [ ]:
def similitud_coseno(v1, v2):
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    return float(np.dot(v1, v2) / denom) if denom > 0 else 0.0


def similitud_jaccard(bolsa1, bolsa2):
    s1    = {p for p, _ in bolsa1}
    s2    = {p for p, _ in bolsa2}
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0.0


def similitud_minmax(v1, v2):
    num = float(np.sum(np.minimum(v1, v2)))
    den = float(np.sum(np.maximum(v1, v2)))
    return num / den if den > 0 else 0.0

## 11. Comparación Directa de Documentos

In [ ]:
i, j = 0, 1
b1, b2   = conteodocs[i], conteodocs[j]
v1_tf    = bolsa_a_vector_tfidf(b1, vocab_idx, vocabulario)
v2_tf    = bolsa_a_vector_tfidf(b2, vocab_idx, vocabulario)
v1_cont  = bolsa_a_vector_conteo(b1, vocab_idx)
v2_cont  = bolsa_a_vector_conteo(b2, vocab_idx)

print(f'Doc {i} → {categorias[etiquetas[i]]}')
print(f'Doc {j} → {categorias[etiquetas[j]]}')
print()
print(f'Coseno  (TF-IDF) : {similitud_coseno(v1_tf,   v2_tf):.4f}')
print(f'Jaccard          : {similitud_jaccard(b1, b2):.4f}')
print(f'MinMax  (conteo) : {similitud_minmax(v1_cont, v2_cont):.4f}')

## 12. Búsqueda por Fuerza Bruta

La búsqueda por fuerza bruta compara la consulta contra **todos** los documentos del corpus.

Dado que `tfidf_norm` ya está pre-normalizada, el coseno se reduce a un producto matricial:

$$\text{sims} = \mathbf{M}_{\text{norm}} \cdot \hat{q}$$

In [ ]:
def busqueda_fuerza_bruta(consulta_bolsa, tfidf_norm, vocab_idx, vocabulario,
                          top_k=5, excluir_idx=None):
    v_q = bolsa_a_vector_tfidf(consulta_bolsa, vocab_idx, vocabulario)
    norm_q = np.linalg.norm(v_q)
    if norm_q == 0:
        return []
    v_q_norm = (v_q / norm_q).astype(np.float32)
    sims = tfidf_norm @ v_q_norm
    if excluir_idx is not None:
        sims[excluir_idx] = -1.0
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(int(i), float(sims[i])) for i in top_idx]

In [ ]:
# Búsqueda usando un documento del corpus como consulta
idx_consulta = 42
resultados_fb = busqueda_fuerza_bruta(
    conteodocs[idx_consulta], tfidf_norm, vocab_idx, vocabulario,
    excluir_idx=idx_consulta
)

print(f'Consulta (doc {idx_consulta}) – {categorias[etiquetas[idx_consulta]]}')
print(f'Texto   : {orig[idx_consulta][:300]}...')
print()
print('Top 5 documentos más similares:')
for rank, (doc_id, sim) in enumerate(resultados_fb, 1):
    print(f'  {rank}. doc {doc_id:>5} | sim={sim:.4f} | {categorias[etiquetas[doc_id]]}')
print()
mejor_id, mejor_sim = resultados_fb[0]
print(f'Documento recuperado (doc {mejor_id}):')
print(orig[mejor_id][:400] + '...')

In [ ]:
def procesar_consulta(texto, datitos, vocab_idx):
    tokens = mochar(lematizar(texto), datitos)
    conteo = Counter(tokens)
    return [(p, c) for p, c in conteo.items() if p in vocab_idx]


consultas = [
    'nasa space mission satellite',
    'government crime enforcement security',
    'computer graphics image rendering',
]

print('=== Búsqueda por fuerza bruta – consultas textuales ===')
for consulta in consultas:
    bolsa_q   = procesar_consulta(consulta, datitos, vocab_idx)
    resultados = busqueda_fuerza_bruta(bolsa_q, tfidf_norm, vocab_idx, vocabulario)
    print(f'\nConsulta: "{consulta}"')
    for rank, (doc_id, sim) in enumerate(resultados, 1):
        print(f'  {rank}. doc {doc_id:>5} | sim={sim:.4f} | {categorias[etiquetas[doc_id]]}')

## 13. Índice Inverso

El índice inverso invierte la relación documento → palabras:

$$\text{palabra} \longrightarrow [\text{doc}_1,\ \text{doc}_2,\ \ldots]$$

Esto permite encontrar documentos candidatos sin recorrer todo el corpus.

In [ ]:
def construir_indice_inverso(conteodocs):
    indice = defaultdict(list)
    for doc_id, bolsa in enumerate(conteodocs):
        for palabra, _ in bolsa:
            indice[palabra].append(doc_id)
    return dict(indice)


indice_inverso = construir_indice_inverso(conteodocs)

print(f'Palabras indexadas: {len(indice_inverso):,}')

# Ejemplo de entrada del índice
palabra_ej = palabras_max[50]
docs_ej    = indice_inverso[palabra_ej]
print(f'\nEjemplo → "{palabra_ej}" aparece en {len(docs_ej):,} documentos')
print(f'Primeros 10 doc_ids: {docs_ej[:10]}')

## 14. Búsqueda Acelerada con Índice Inverso

Flujo de búsqueda acelerada:

1. Consulta → índice inverso → **documentos candidatos** (unión de posting lists)
2. Calcular similitud **solo** contra candidatos
3. Ordenar y devolver top-k

Se evita calcular similitud contra documentos que no comparten ningún término con la consulta.

In [ ]:
def busqueda_indice_inverso(consulta_bolsa, indice, tfidf_norm, vocab_idx, vocabulario, top_k=5):
    candidatos = set()
    for p, _ in consulta_bolsa:
        if p in indice:
            candidatos.update(indice[p])
    candidatos = list(candidatos)
    if not candidatos:
        return [], 0

    v_q = bolsa_a_vector_tfidf(consulta_bolsa, vocab_idx, vocabulario)
    norm_q = np.linalg.norm(v_q)
    if norm_q == 0:
        return [], len(candidatos)
    v_q_norm = (v_q / norm_q).astype(np.float32)

    sims      = tfidf_norm[candidatos] @ v_q_norm
    top_local = np.argsort(sims)[::-1][:top_k]
    return [(candidatos[k], float(sims[k])) for k in top_local], len(candidatos)

In [ ]:
print('=== Búsqueda con índice inverso – mismas consultas ===')
for consulta in consultas:
    bolsa_q             = procesar_consulta(consulta, datitos, vocab_idx)
    resultados, n_cand  = busqueda_indice_inverso(
        bolsa_q, indice_inverso, tfidf_norm, vocab_idx, vocabulario
    )
    pct = n_cand / len(conteodocs) * 100
    print(f'\nConsulta: "{consulta}"')
    print(f'Candidatos revisados: {n_cand:,} / {len(conteodocs):,} ({pct:.1f}%)')
    for rank, (doc_id, sim) in enumerate(resultados, 1):
        print(f'  {rank}. doc {doc_id:>5} | sim={sim:.4f} | {categorias[etiquetas[doc_id]]}')

## 15. Comparación: Fuerza Bruta vs Índice Inverso

In [ ]:
stats = []

for consulta in consultas:
    bolsa_q = procesar_consulta(consulta, datitos, vocab_idx)

    t0     = time.time()
    res_fb = busqueda_fuerza_bruta(bolsa_q, tfidf_norm, vocab_idx, vocabulario)
    t_fb   = time.time() - t0

    t0             = time.time()
    res_ii, n_cand = busqueda_indice_inverso(
        bolsa_q, indice_inverso, tfidf_norm, vocab_idx, vocabulario
    )
    t_ii = time.time() - t0

    top_fb = res_fb[0][0] if res_fb else None
    top_ii = res_ii[0][0] if res_ii else None

    stats.append({
        'consulta': consulta, 't_fb': t_fb, 't_ii': t_ii,
        'cand_fb': len(conteodocs), 'cand_ii': n_cand,
        'mismo_top': top_fb == top_ii
    })

col = f'{"Consulta":<38} {"T.FB(s)":>8} {"T.II(s)":>8} {"Docs FB":>8} {"Cands II":>9} {"=Top?":>6}'
print(col)
print('-' * len(col))
for s in stats:
    print(f'{s["consulta"]:<38} {s["t_fb"]:>8.4f} {s["t_ii"]:>8.4f} '
          f'{s["cand_fb"]:>8,} {s["cand_ii"]:>9,} {str(s["mismo_top"]):>6}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels   = [f'"{c[:22]}..."' if len(c) > 22 else f'"{c}"' for c in consultas]
cand_fb  = [s['cand_fb'] for s in stats]
cand_ii  = [s['cand_ii'] for s in stats]
t_fb_lst = [s['t_fb']    for s in stats]
t_ii_lst = [s['t_ii']    for s in stats]
x, w     = range(len(consultas)), 0.35

# Documentos revisados
ax = axes[0]
ax.bar([i - w/2 for i in x], cand_fb, w, label='Fuerza Bruta', color='steelblue')
ax.bar([i + w/2 for i in x], cand_ii, w, label='Índice Inverso', color='tomato')
ax.set_title('Documentos revisados por consulta')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Documentos')
ax.legend()

# Tiempo de ejecución
ax = axes[1]
ax.bar([i - w/2 for i in x], t_fb_lst, w, label='Fuerza Bruta', color='steelblue')
ax.bar([i + w/2 for i in x], t_ii_lst, w, label='Índice Inverso', color='tomato')
ax.set_title('Tiempo de ejecución por consulta')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Segundos')
ax.legend()

plt.tight_layout()
plt.show()